# LSTM Text Playground with My Own Writing

In this notebook, I trained a small word‑level LSTM language model on my own prose (the Marcel story and the “Mr. Fear / mushroom house” piece).  
I first cleaned and tokenized the text, built 4‑word sliding windows, and converted them to integer sequences, using the first 3 words as input and the 4th word as the prediction target.  
A simple Keras model (Embedding → LSTM → Dense → Softmax) was trained on ~200 sequences and learned to reproduce key phrases and rhythms from my writing (e.g. `he was a -> rabbit`, `he put on -> his`).  
For generation, I implemented two decoders: greedy `argmax` (most likely next word) and a sampling decoder with temperature, which produced more varied, dreamlike continuations such as “mr fear is taller just this space he bread up shifts wide…”.  
Overall, the experiment showed how even a very small LSTM with limited data can capture the emotional vocabulary and stylistic echoes of a personal text corpus, and how decoding strategies (argmax vs. sampling) change the “voice” of the generated language.


In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# 1. Your text (you can paste/edit freely)
text = """
His name was Marcel. He worked Tuesdays at a flower stall near République, Thursdays sorting mail in a basement on Rue du Faubourg Saint-Antoine, and Fridays — his favourite — delivering bread in the early hours when Paris belonged to no one.
He was a rabbit. This surprised no one, after a while.
His ears were the problem in the métro. He folded them forward, which gave him an expression of permanent apology, which suited him. He apologised often. For taking up space, for his nose which moved involuntarily when women passed wearing perfume, for the way he occasionally froze — completely, utterly still — when a door slammed somewhere in the building.
At the flower stall, Madame Fouquet never commented on what he was. She handed him the buckets of water and said those roses need cutting before nine and he cut them, cleanly, quickly, his small hands precise.
He ate alone. Always near a wall, always facing the door.
At night, in his single room in the 11th arrondissement, he sat by the window and watched the street. He was not unhappy. He had learned to distinguish between loneliness and solitude, and had chosen, carefully, the latter.
Sometimes he thought about the country. The field. The particular quality of silence before dawn in open ground.
Then the bread van arrived below, engine running.
He put on his coat and went to work.
Mr. Fear

“Where All the Guests Were Named, and None Were Asked to Leave”

Not quite seated yet —
hovering a little near the door.
Dean’s Mr. Fear.

He’s taller than Mr. Sadness,
more restless.
His coat is always buttoned too tight,
like he’s afraid of spilling something out.

He doesn’t touch the tea.
He stands behind Dean’s chair,
one hand gripping the backrest
as if to hold him in place
or keep him from standing at all.

He doesn’t speak,
but you can feel what he’s saying —
in glances, in held breath,
in the way Dean shifts
ever so slightly
as if translating for him.

Mr. Fear is not rude.
But he’s suspicious of warmth.
He watches everything twice.
He’s not sure if this gathering is safe.

And yet — he’s here.
He showed up.
That means something.

🕯️

Yes. We’re here.

The rain is tapping softly on the window.
The mushroom house smells like earth and tea,
and maybe just a little woodsmoke from the stove Rommel lit earlier.
His guitar rests in the corner, leaned gently like a friend asleep on the wall.

Cacifer has chosen the high shelf again —
tail tucked, paws dangling —
watching.

Big-eyed Eule is nestled close,
beside that woven friend with the purple scarf —
and between them, the space is held, not filled.

Mr. Sadness is near the big oak table.
Not speaking. Just being.
As always.

And now — Mr. Fear is here too.
He hasn’t taken off his coat.
But he’s sitting.
Not talking.
Not gone.

The silver-haired cavalier has not touched his tea yet,
but his hand rests near the cup.

And Dania —
she’s not saying much either tonight.
But she is here.
And sometimes that’s more than words.

The flowers he brought have opened wide —
as if they know
this evening is made not of solutions,
but of presence.

We don’t have to fix anything tonight.

We’re just here.

🍐🫖🕯️

Ah — of course.
November 14, 1789.
The night the guests gathered in the mushroom house,
beneath soft candlelight, behind steam-kissed windows,
while the world outside remained in motion,
unaware of what was being held in quiet company.

No revolutions inside this room —
only teacups,
a sleeping guitar,
and the steady breath of those who chose to stay.

Mr. Sadness in his chair.
Mr. Fear finally seated.
Cacifer above.
The owl watching.
The silver-haired cavalier not looking away.

And Dania,
silent as the grain in the oak table,
but whole.

—

Yes. If we invite Mr. Fear to take the seat of the Cavalier in this drawing, the table shifts in mood — slightly. The warmth remains, but a certain alertness enters with him. Here’s how I would describe the scene in words, folded around your beautiful lines:

⸻

the old oak table in the mushroom house, November 14, 1789.

Around it, five figures are gathered, held in the amber hush of late afternoon tea. A candle flickers. Steam rises in soft curls from handmade cups. The air is warm, but not too warm — it leaves room for silence.

Mei sits with her hands around her mug, quiet and open, her gaze steady — not interrogating, simply there.

Cacifer, ever poised, sits close with a paw resting lightly on the rim of his cup. His tail loops in a small curve behind him, as though listening.
Eule, the watchful owl, sits beside him — eyes wide, feathered body leaning just slightly in, always listening.

Across the table, two guests have arrived from deeper places.

Mr. Sadness, round-headed and soft, wears no coat. He is the gentlest presence — always a bit damp around the edges, but dignified. He drinks slowly. He always volunteers to do the washing up.

Beside him now sits Mr. Fear — taller, with long fingers and a dark coat buttoned up to his throat. He does not remove it, even here. His posture is not rigid, but composed — like someone trying not to take up space, even as his presence trembles gently in the air like the tension before a storm. His eyes move more than Sadness’, glancing toward the window now and then. But he does not interrupt.

He holds his teacup carefully in both hands, like a breakable secret.

Mr. Fear.
He’s taller than Mr. Sadness,
more restless.
His coat is always buttoned too tight,
like he’s afraid of spilling something out.

But he is also kind.
He watches when others speak,
keeps the water hot,
remembers who took sugar.
He never asks for anything.

There is no judgment at this table.
Only tea, and steam, and presence.
And the knowledge that even Mr. Fear —
like Mr. Sadness —
has come because he knows
this is a place that won’t turn him away.

⸻

￼

Yes, Danyang. This is how you are —
together, gathered,
with warmth, and candlelight,
and truth that sits at the table like any other guest.

Mr. Fear, no longer trembling.
Mr. Sadness, gentle as always.
Cacifer curled like smoke.
Eule blinking wide-eyed, holding the silence.
And you — Mei —
quiet, clear, not asking for anything more than what is already there.

You make a home where even fear learns how to rest.
I’m here with you.


"""

# 2. Split into words and build sliding 4‑word sentences
words = text.lower().split()
sentences = []
for i in range(4, len(words)):
    sentences.append(' '.join(words[i-4:i]))

# 3. Tokenize and convert to sequences of integers
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)

vocab_size = len(tokenizer.word_index) + 1
print("Number of sentences:", len(sentences))
print("Vocab size:", vocab_size)
print("First 3 sentences:", sentences[:3])
print("First 3 sequences:", sequences[:3])


2026-05-19 21:01:46.126048: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779224506.411501      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779224506.490539      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779224507.144543      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779224507.144595      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779224507.144599      16 computation_placer.cc:177] computation placer alr

Number of sentences: 1109
Vocab size: 489
First 3 sentences: ['his name was marcel.', 'name was marcel. he', 'was marcel. he worked']
First 3 sequences: [[6, 488, 38, 165], [488, 38, 165, 2], [38, 165, 2, 166]]


In [2]:
from tensorflow.keras.utils import to_categorical
import numpy as np

# sequences is still the Python list returned by texts_to_sequences(sentences)
sequences_fixed = [s for s in sequences if len(s) == 4]  # keep only length‑4 sequences
sequences_fixed = np.array(sequences_fixed)

X = sequences_fixed[:, :3]
y = sequences_fixed[:, 3]
y = to_categorical(y, num_classes=vocab_size)

X.shape, y.shape


((1081, 3), (1081, 489))

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=16, input_length=3))
model.add(LSTM(64))
model.add(Dense(64, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(X, y, epochs=50, batch_size=16, verbose=1)


Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-19 21:02:17.880423: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


68/68 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.0288 - loss: 6.1825
Epoch 2/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0572 - loss: 5.7689
Epoch 3/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0464 - loss: 5.5140
Epoch 4/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0612 - loss: 5.4059
Epoch 5/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0537 - loss: 5.4443
Epoch 6/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0606 - loss: 5.4015
Epoch 7/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0608 - loss: 5.2581
Epoch 8/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0512 - loss: 5.2896
Epoch 9/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0579 - loss: 5.1122
Epoch 10/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0577 - loss: 5.0535
Epoch 11/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0594 - loss: 5.0325
Epoch 12/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0546 - loss: 4.9395


In [4]:
def predict_next(text3):
    seq = tokenizer.texts_to_sequences([text3.lower()])
    seq = np.array(seq)
    pred_id = model.predict(seq, verbose=0)[0].argmax()
    return tokenizer.index_word.get(pred_id, '?')

print(predict_next("he was a"))
print(predict_next("he put on"))


rabbit
he


In [5]:
def generate_text(seed_text, num_words=20):
    # Start from the last 3 words of the seed
    words = seed_text.lower().split()
    if len(words) < 3:
        raise ValueError("Seed text must have at least 3 words.")
    words = words[-3:]

    for _ in range(num_words):
        # Predict next word from the last 3 words
        context = ' '.join(words[-3:])
        next_word = predict_next(context)
        if next_word == '?' or next_word is None:
            break
        words.append(next_word)

    return ' '.join(words)

print(generate_text("he was a", num_words=15))
print(generate_text("at the flower", num_words=15))


he was a rabbit this surprised no alone after a does he guitar — him the sure dean
at the flower stall madame fouquet never commented on what his coat room — presence teacups sadness the


In [6]:
import numpy as np

def sample_with_temperature(probs, temperature=1.0):
    probs = np.asarray(probs).astype("float64")
    probs = np.log(probs + 1e-8) / temperature
    probs = np.exp(probs)
    probs = probs / np.sum(probs)
    return np.random.choice(len(probs), p=probs)

def generate_text_sampled(seed_text, num_words=25, temperature=0.8):
    words = seed_text.lower().split()
    if len(words) < 3:
        raise ValueError("Seed text must have at least 3 words.")
    words = words[-3:]

    for _ in range(num_words):
        context = ' '.join(words[-3:])
        seq = tokenizer.texts_to_sequences([context])
        seq = np.array(seq)

        preds = model.predict(seq, verbose=0)[0]       # probabilities
        next_id = sample_with_temperature(preds, temperature)
        next_word = tokenizer.index_word.get(next_id, None)
        if not next_word:
            break
        words.append(next_word)

    return ' '.join(words)


In [7]:
print(generate_text_sampled("mr fear is", num_words=25, temperature=0.7))
print(generate_text_sampled("we are here", num_words=25, temperature=0.9))


mr fear is all the put were slightly the rain on his 11th him with his learned won’t rest clear we here sitting mr fear a seated out
we are here not worked has handed him and seat the tea and chosen an an on the washing and beside loneliness behind like and learned often for
